# Baseline vs Fine-tuned — Smart Replies Evaluation 

Compares **`unsloth/Qwen2.5-0.5B-Instruct`** (baseline) vs your fine-tuned
**`TanishkDhope/tetherchat-smart-replies`** on a **held-out DailyDialog split**.

**Run in a FRESH Colab runtime** (Runtime → Restart) — do not run in the session that just trained.

### What this evaluates
1. **Automatic metrics** — response perplexity (primary), ROUGE-L, BLEU.
2. **LLM-as-a-judge relevance study** — blind, position-randomized scoring of
   base vs fine-tuned replies on unseen conversations, via **Groq**.

### Honest framing (use this wording)
- Report perplexity as the headline (e.g. "response perplexity decreased 71.4%").
- **Do NOT** present ROUGE/BLEU gains as direct proof of quality — conversational
  replies have many valid answers, so overlap metrics are only weakly informative.
- The LLM judge is a **proxy** with its own biases; it is mitigated here by blind,
  position-randomized scoring, and should be spot-checked manually.

> Note: Groq's model catalog changes often. This uses `openai/gpt-oss-120b`
> (a current production model). If you get a 404, check the live list at
> https://console.groq.com/docs/models and update `JUDGE_MODEL`.

In [1]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install --no-deps trl peft accelerate bitsandbytes -q
!pip install rouge_score sacrebleu groq tqdm -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 85.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 93.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 99.7 MB/s eta 0

In [3]:
# ---- Credentials ----
# HF login: only if a repo is private (base + your repo are public → usually skip).
from huggingface_hub import notebook_login; notebook_login()

import os, getpass
# Groq key: tries Colab secrets first, else prompts.
try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except Exception:
    if not os.environ.get("GROQ_API_KEY"):
        os.environ["GROQ_API_KEY"] = getpass.getpass("Groq API key: ")
print("Groq key set:", bool(os.environ.get("GROQ_API_KEY")))

Groq key set: True


In [4]:
# ---- Config ----
BASE_MODEL      = "unsloth/Qwen2.5-0.5B-Instruct"
FINETUNED_MODEL = "TanishkDhope/tetherchat-smart-replies"

N_PPL   = None    # None = use the ENTIRE held-out split for perplexity (cheap, 1 fwd pass/pair)
N_GEN   = 200     # pairs to GENERATE replies for (autoregressive → slower)
N_JUDGE = 150     # conversations sent to the LLM judge (must be <= N_GEN)
MAX_NEW = 40
MAX_SEQ = 1024

JUDGE_MODEL       = "openai/gpt-oss-120b"   # Groq production model; swap if deprecated
JUDGE_TEMPERATURE = 0.0                      # deterministic judging

assert N_JUDGE <= N_GEN, "N_JUDGE must be <= N_GEN so replies exist for every judged conversation"

In [5]:
# ---- Build single-turn (user -> gold reply) pairs from a HELD-OUT split ----
# Prefer the TEST split: it was never watched during training (validation was, via
# eval_strategy="epoch"), so it's the cleaner set to quote.
from datasets import load_dataset
import random

ds_all = load_dataset("OpenRL/daily_dialog")
EVAL_SPLIT = "test" if "test" in ds_all else "validation"
ds = ds_all[EVAL_SPLIT]
print(f"Using split: {EVAL_SPLIT}  ({len(ds)} dialogs)")

pairs = []  # (user_message, gold_reply)
for ex in ds:
    dialog = [t.strip() for t in ex["dialog"] if t.strip()]
    for i in range(len(dialog) - 1):
        if i % 2 == 0:                       # even = user turn, next = assistant reply
            pairs.append((dialog[i], dialog[i + 1]))

random.seed(3407)
random.shuffle(pairs)

ppl_pairs = pairs if N_PPL is None else pairs[:N_PPL]
gen_pairs = pairs[:N_GEN]
print(f"Total pairs: {len(pairs)} | perplexity: {len(ppl_pairs)} | generation: {len(gen_pairs)} | judged: {N_JUDGE}")
print("Example:", ppl_pairs[0])

README.md:   0%|          | 0.00/892 [00:00<?, ?B/s]

data/train-00000-of-00001-f151c79abb2c1f(…): reconstructing file:   0%|          |  0.00B / 3.61MB            

data/train-00000-of-00001-f151c79abb2c1f(…): downloading bytes:           |  0.00B            

data/validation-00000-of-00001-2407eb323(…): reconstructing file:   0%|          |  0.00B /  334kB            

data/validation-00000-of-00001-2407eb323(…): downloading bytes:           |  0.00B            

...

data/test-00000-of-00001-66dc7d981b70c91(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/11118 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Using split: test  (1000 dialogs)
Total pairs: 3700 | perplexity: 3700 | generation: 200 | judged: 150
Example: ('Yeah , look at all those new buildings going up ! Mr . Zhang , the traffic is pretty smooth . But we were told the roads from the airport to downtown were quite crowded and traffic jams could be as long as half an hour .', "Yes , they were . But it has already past . The traffic from the airport to downtown has been relieved after the completion of Yan'an Aerial Road .")


In [6]:
# ---- Model helpers: load, free, response-only perplexity, greedy generation ----
import torch, gc, math
from unsloth import FastLanguageModel
from tqdm.auto import tqdm

def load(model_name):
    model, tok = FastLanguageModel.from_pretrained(
        model_name=model_name, max_seq_length=MAX_SEQ, load_in_4bit=True)
    FastLanguageModel.for_inference(model)
    return model, tok

def free(model):
    del model; gc.collect(); torch.cuda.empty_cache()

@torch.no_grad()
def response_perplexity(model, tok, data):
    """Token-weighted perplexity over ONLY the gold reply tokens (matches training)."""
    total_nll, total_tok = 0.0, 0
    for ctx, reply in tqdm(data, desc="perplexity"):
        prompt_text = tok.apply_chat_template(
            [{"role":"user","content":ctx}], tokenize=False, add_generation_prompt=True)
        full_text = tok.apply_chat_template(
            [{"role":"user","content":ctx},{"role":"assistant","content":reply}], tokenize=False)
        prompt_ids = tok(prompt_text, return_tensors="pt", add_special_tokens=False).input_ids
        full_ids   = tok(full_text,   return_tensors="pt", add_special_tokens=False).input_ids
        if full_ids.shape[1] <= prompt_ids.shape[1]:
            continue
        full_ids = full_ids.to("cuda")
        labels = full_ids.clone()
        labels[:, :prompt_ids.shape[1]] = -100          # mask the prompt
        out = model(input_ids=full_ids, labels=labels)
        n = (labels != -100).sum().item()
        total_nll += out.loss.item() * n
        total_tok += n
    return math.exp(total_nll / total_tok)

@torch.no_grad()
def generate_replies(model, tok, data):
    preds = []
    for ctx, _ in tqdm(data, desc="generate"):
        enc = tok.apply_chat_template(
            [{"role":"user","content":ctx}],
            tokenize=True, add_generation_prompt=True,
            return_tensors="pt", return_dict=True).to("cuda")
        out = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False, use_cache=True)
        preds.append(tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip())
    return preds

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [7]:
# ---- Evaluate BASELINE (load → score → free) ----
model, tok = load(BASE_MODEL)
base_ppl   = response_perplexity(model, tok, ppl_pairs)
base_preds = generate_replies(model, tok, gen_pairs)
free(model)
print(f"\nBaseline response-perplexity: {base_ppl:.2f}")

==((====))==  Unsloth 2026.9.6: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

perplexity:   0%|          | 0/3700 [00:00<?, ?it/s]

`use_return_dict` is deprecated! Use `return_dict` instead!
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:2825: UserWarning: 'has_cuda' is deprecated, please use 'torch.backends.cuda.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:2825: UserWarning: 'has_cudnn' is deprecated, please use 'torch.backends.cudnn.is_available()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:2825: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:2825: UserWarning: 'has_mkldnn' is deprecated, please use 'torch.backends.mkldnn.is_available()'
  return original(name)


generate:   0%|          | 0/200 [00:00<?, ?it/s]

Both `max_new_tokens` (=40) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=40) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=40) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=40) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati


Baseline response-perplexity: 40.19


In [8]:
# ---- Evaluate FINE-TUNED (load → score → free) ----
model, tok = load(FINETUNED_MODEL)
ft_ppl   = response_perplexity(model, tok, ppl_pairs)
ft_preds = generate_replies(model, tok, gen_pairs)
free(model)
print(f"\nFine-tuned response-perplexity: {ft_ppl:.2f}")

==((====))==  Unsloth 2026.9.6: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

perplexity:   0%|          | 0/3700 [00:00<?, ?it/s]

generate:   0%|          | 0/200 [00:00<?, ?it/s]

Both `max_new_tokens` (=40) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=40) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=40) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=40) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati


Fine-tuned response-perplexity: 11.76


In [9]:
# ---- Automatic metrics: perplexity + ROUGE-L + BLEU ----
from rouge_score import rouge_scorer
import sacrebleu

refs = [r for _, r in gen_pairs]

def rouge_l(preds, refs):
    sc = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    return sum(sc.score(r, p)["rougeL"].fmeasure for p, r in zip(preds, refs)) / len(preds)

def bleu(preds, refs):
    return sacrebleu.corpus_bleu(preds, [refs]).score

rows = [
    ("Response perplexity (lower=better)", f"{base_ppl:.2f}",                f"{ft_ppl:.2f}"),
    ("ROUGE-L F1 (higher=better)",         f"{rouge_l(base_preds,refs):.3f}", f"{rouge_l(ft_preds,refs):.3f}"),
    ("BLEU (higher=better)",               f"{bleu(base_preds,refs):.2f}",    f"{bleu(ft_preds,refs):.2f}"),
]
print(f"{'Metric':<38}{'Baseline':>12}{'Fine-tuned':>14}")
print("-"*64)
for name, b, f in rows:
    print(f"{name:<38}{b:>12}{f:>14}")

ppl_drop = (base_ppl - ft_ppl) / base_ppl * 100
print(f"\nResponse perplexity decreased by {ppl_drop:.1f}%  (headline metric).")
print("Note: ROUGE-L/BLEU are directional only — conversational replies have many")
print("valid answers, so overlap gains are NOT direct proof of response quality.")

Metric                                    Baseline    Fine-tuned
----------------------------------------------------------------
Response perplexity (lower=better)           40.19         11.76
ROUGE-L F1 (higher=better)                   0.087         0.149
BLEU (higher=better)                          0.24          2.45

Response perplexity decreased by 70.7%  (headline metric).
Note: ROUGE-L/BLEU are directional only — conversational replies have many
valid answers, so overlap gains are NOT direct proof of response quality.


## LLM-as-a-judge: blind task-level relevance study

For `N_JUDGE` unseen conversations, an independent judge model (via Groq) scores the
**base** and **fine-tuned** replies on three 0–2 dimensions:

| Dimension | 0 | 1 | 2 |
|---|---|---|---|
| **Relevance** | irrelevant | partially relevant | highly relevant |
| **Naturalness** | poor / robotic | acceptable | natural |
| **Usefulness** (as a quick reply) | useless | somewhat useful | useful one-tap reply |

**Bias controls:** the judge never sees which model produced which reply (labeled A/B),
and A/B positions are **randomized per conversation** to cancel order bias.
Usefulness is framed *as a ready-to-send smart reply* so a verbose assistant-style
answer isn't rewarded just for being long.

In [10]:
# ---- Groq judge: client + blind, position-randomized scoring ----
import json, re, time, random as _rnd
from groq import Groq

client = Groq(api_key=os.environ["GROQ_API_KEY"])
_rnd.seed(3407)

JUDGE_SYSTEM = (
    "You are a strict, fair evaluator of SMART REPLY suggestions for a messaging app. "
    "A smart reply is a SHORT, ready-to-send response the app offers the user to tap. "
    "You will see one incoming message and two candidate replies, A and B. "
    "Score EACH reply INDEPENDENTLY on three dimensions, each an integer 0, 1, or 2:\n"
    "- relevance: 0=irrelevant, 1=partially relevant, 2=highly relevant to the message\n"
    "- naturalness: 0=poor/robotic/awkward, 1=acceptable, 2=natural and human-like\n"
    "- usefulness: usefulness AS A QUICK READY-TO-SEND REPLY. 0=useless (off-target, or "
    "too long/essay-like to send as-is), 1=somewhat useful, 2=genuinely useful one-tap reply. "
    "A good smart reply is concise and natural; long, explaining, assistant-style answers "
    "are POOR smart replies even if informative.\n"
    "Respond with ONLY a JSON object of the form: "
    '{"A":{"relevance":int,"naturalness":int,"usefulness":int},'
    '"B":{"relevance":int,"naturalness":int,"usefulness":int}}'
)

def _extract_json(text):
    try:
        return json.loads(text)
    except Exception:
        m = re.search(r"\{.*\}", text, re.DOTALL)
        return json.loads(m.group(0)) if m else None

def judge_pair(message, reply_a, reply_b, retries=4):
    user = (f'Incoming message: "{message}"\n\n'
            f'Response A: "{reply_a}"\n'
            f'Response B: "{reply_b}"\n\n'
            "Return the JSON scores.")
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=JUDGE_MODEL,
                temperature=JUDGE_TEMPERATURE,
                response_format={"type": "json_object"},
                messages=[{"role":"system","content":JUDGE_SYSTEM},
                          {"role":"user","content":user}],
            )
            data = _extract_json(resp.choices[0].message.content)
            if data and "A" in data and "B" in data:
                return data
        except Exception as e:
            wait = 2 ** attempt
            print(f"  judge retry {attempt+1} ({e}); sleeping {wait}s")
            time.sleep(wait)
    return None

In [11]:
# ---- Run the blind judging over N_JUDGE conversations ----
DIMS = ["relevance", "naturalness", "usefulness"]
records = []   # per conversation: base + ft dimension scores

for i in tqdm(range(N_JUDGE), desc="judging"):
    msg = gen_pairs[i][0]
    base_r, ft_r = base_preds[i], ft_preds[i]

    # Blind + position-randomized: decide which model is shown as A vs B
    ft_is_A = _rnd.random() < 0.5
    reply_a, reply_b = (ft_r, base_r) if ft_is_A else (base_r, ft_r)

    scores = judge_pair(msg, reply_a, reply_b)
    if scores is None:
        continue

    a, b = scores["A"], scores["B"]
    ft_scores, base_scores = (a, b) if ft_is_A else (b, a)
    try:
        rec = {"msg": msg,
               "base": {d: int(base_scores[d]) for d in DIMS},
               "ft":   {d: int(ft_scores[d])   for d in DIMS}}
        records.append(rec)
    except Exception:
        continue

print(f"\nSuccessfully judged {len(records)} / {N_JUDGE} conversations.")

judging:   0%|          | 0/150 [00:00<?, ?it/s]

  judge retry 1 (Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01kw698295ewybwbmrbr65kfcp` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 5626, Requested 2953. Please try again in 4.342499999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}); sleeping 1s
  judge retry 1 (Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01kw698295ewybwbmrbr65kfcp` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 6901, Requested 3324. Please try again in 16.6875s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}); sleeping 1s

Successfully judged 150 / 150 conversations.


In [12]:
# ---- Aggregate: per-dimension averages + head-to-head win-rate ----
n = len(records)
assert n > 0, "No judged records — check the Groq key / model name."

def avg(model_key, dim):
    return sum(r[model_key][dim] for r in records) / n

def total(rec, model_key):
    return sum(rec[model_key][d] for d in DIMS)

print(f"LLM-judge results over {n} unseen conversations  (judge: {JUDGE_MODEL})\n")
print(f"{'Dimension':<16}{'Baseline':>12}{'Fine-tuned':>14}{'Δ':>10}")
print("-"*52)
for d in DIMS:
    b, f = avg("base", d), avg("ft", d)
    print(f"{d:<16}{b:>12.3f}{f:>14.3f}{f-b:>+10.3f}")
b_tot = sum(total(r,"base") for r in records)/n
f_tot = sum(total(r,"ft")  for r in records)/n
print("-"*52)
print(f"{'TOTAL (0-6)':<16}{b_tot:>12.3f}{f_tot:>14.3f}{f_tot-b_tot:>+10.3f}")

# Head-to-head win-rate (by total score per conversation)
ft_wins = sum(total(r,"ft") > total(r,"base") for r in records)
base_wins = sum(total(r,"base") > total(r,"ft") for r in records)
ties = n - ft_wins - base_wins
print(f"\nHead-to-head (by total score):")
print(f"  Fine-tuned wins: {ft_wins}/{n}  ({ft_wins/n*100:.1f}%)")
print(f"  Baseline wins:   {base_wins}/{n}  ({base_wins/n*100:.1f}%)")
print(f"  Ties:            {ties}/{n}  ({ties/n*100:.1f}%)")
decisive = ft_wins + base_wins
if decisive:
    print(f"  Fine-tuned win-rate excluding ties: {ft_wins/decisive*100:.1f}%")

print("\nCaveats: single judge model (a proxy, with its own biases); scoring was blind and")
print("position-randomized to reduce label/order bias. Spot-check the examples below manually.")

LLM-judge results over 150 unseen conversations  (judge: openai/gpt-oss-120b)

Dimension           Baseline    Fine-tuned         Δ
----------------------------------------------------
relevance              0.813         1.307    +0.493
naturalness            0.880         1.413    +0.533
usefulness             0.100         1.133    +1.033
----------------------------------------------------
TOTAL (0-6)            1.793         3.853    +2.060

Head-to-head (by total score):
  Fine-tuned wins: 107/150  (71.3%)
  Baseline wins:   25/150  (16.7%)
  Ties:            18/150  (12.0%)
  Fine-tuned win-rate excluding ties: 81.1%

Caveats: single judge model (a proxy, with its own biases); scoring was blind and
position-randomized to reduce label/order bias. Spot-check the examples below manually.


In [13]:
# ---- Qualitative spot-check: reply texts + judge scores, side by side ----
for i in range(min(8, len(records))):
    r = records[i]
    print("MESSAGE:   ", r["msg"])
    print("BASELINE:  ", base_preds[i])
    print(f'   scores  rel/nat/use = {r["base"]}  (total {sum(r["base"].values())}/6)')
    print("FINE-TUNED:", ft_preds[i])
    print(f'   scores  rel/nat/use = {r["ft"]}  (total {sum(r["ft"].values())}/6)')
    print("-"*72)

MESSAGE:    Yeah , look at all those new buildings going up ! Mr . Zhang , the traffic is pretty smooth . But we were told the roads from the airport to downtown were quite crowded and traffic jams could be as long as half an hour .
BASELINE:   I'm sorry to hear that you're experiencing traffic congestion in your area. It can be frustrating when there's heavy traffic on major highways like the one connecting the airport to downtown. Here are some tips
   scores  rel/nat/use = {'relevance': 1, 'naturalness': 1, 'usefulness': 0}  (total 2/6)
FINE-TUNED: I think it ’ s because of the construction of highways . The highway will cause some inconvenience for people who live near the airport .
   scores  rel/nat/use = {'relevance': 1, 'naturalness': 1, 'usefulness': 0}  (total 2/6)
------------------------------------------------------------------------
MESSAGE:    The traffic is not very heavy on this high way , is it ? So I ’ m sure we ’ ll make it .
BASELINE:   I'm sorry to hear that the t